# Custom Middleware

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will learn how to build custom middleware with hook points, short-circuiting, and stacking for LangChain agents.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph langchain-mcp-adapters

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic Logging Middleware

Create middleware that logs input and output using `before_agent` and `after_agent` hooks.

In [ ]:
from langgraph.prebuilt.middleware import AgentMiddleware

class LoggingMiddleware(AgentMiddleware):
    def before_agent(self, state):
        print(f"[INPUT] {state['messages'][-1].content}")
        return state

    def after_agent(self, state):
        print(f"[OUTPUT] {state['messages'][-1].content}")
        return state

print("LoggingMiddleware defined.")

## 4. Use Middleware with an Agent

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")

agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[LoggingMiddleware()],
)

result = agent.invoke({
    "messages": [HumanMessage(content="What is the capital of France?")]
})
print(f"\nFinal: {result['messages'][-1].content}")

## 5. Full Lifecycle Hooks

Use all four hook points: `before_agent`, `before_model`, `after_model`, `after_agent`.

In [ ]:
class FullLifecycleMiddleware(AgentMiddleware):
    def before_agent(self, state):
        print("1. Before agent processes input")
        return state

    def before_model(self, state):
        print("2. Before LLM call")
        return state

    def after_model(self, state):
        print("3. After LLM returns")
        return state

    def after_agent(self, state):
        print("4. After agent produces response")
        return state

agent_full = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[FullLifecycleMiddleware()],
)

result = agent_full.invoke({
    "messages": [HumanMessage(content="Hello!")]
})
print(f"\nResponse: {result['messages'][-1].content}")

## 6. @hook_config Decorator

Configure hooks to only process specific message types.

In [ ]:
from langgraph.prebuilt.middleware import hook_config

class SelectiveMiddleware(AgentMiddleware):
    @hook_config(message_types=["human"])
    def before_agent(self, state):
        print(f"[HUMAN INPUT] {state['messages'][-1].content}")
        return state

    @hook_config(message_types=["ai"])
    def after_agent(self, state):
        print(f"[AI OUTPUT] {state['messages'][-1].content}")
        return state

agent_selective = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[SelectiveMiddleware()],
)

result = agent_selective.invoke({
    "messages": [HumanMessage(content="Tell me a fun fact.")]
})
print(f"\nResponse: {result['messages'][-1].content}")

## 7. Short-Circuiting with can_jump_to

Skip agent processing and return cached responses directly.

In [ ]:
class CacheMiddleware(AgentMiddleware):
    def __init__(self):
        self.cache = {}

    def before_agent(self, state):
        user_msg = state["messages"][-1].content
        if user_msg in self.cache:
            print(f"[CACHE HIT] {user_msg}")
            return self.can_jump_to(
                state,
                response=self.cache[user_msg],
            )
        print(f"[CACHE MISS] {user_msg}")
        return state

    def after_agent(self, state):
        user_msg = state["messages"][-2].content
        response = state["messages"][-1].content
        self.cache[user_msg] = response
        return state

cache_mw = CacheMiddleware()

agent_cached = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[cache_mw],
)

result1 = agent_cached.invoke({
    "messages": [HumanMessage(content="What is 2 + 2?")]
})
print(f"First call: {result1['messages'][-1].content}")

result2 = agent_cached.invoke({
    "messages": [HumanMessage(content="What is 2 + 2?")]
})
print(f"Second call (cached): {result2['messages'][-1].content}")

## 8. Timed Logging Middleware

Record request/response pairs with timestamps and elapsed time.

In [ ]:
import time

class TimedLoggingMiddleware(AgentMiddleware):
    def __init__(self):
        self.logs = []

    def before_agent(self, state):
        state["_start_time"] = time.time()
        self.logs.append({
            "type": "input",
            "content": state["messages"][-1].content,
            "timestamp": time.time(),
        })
        return state

    def after_agent(self, state):
        elapsed = time.time() - state.get("_start_time", time.time())
        self.logs.append({
            "type": "output",
            "content": state["messages"][-1].content,
            "timestamp": time.time(),
            "elapsed_seconds": round(elapsed, 2),
        })
        return state

logger = TimedLoggingMiddleware()

agent_timed = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[logger],
)

result = agent_timed.invoke({
    "messages": [HumanMessage(content="Explain middleware in one sentence.")]
})

print("Logs:")
for log in logger.logs:
    print(f"  [{log['type']}] {log['content'][:80]}")

## 9. Stacking Multiple Middleware

Combine middleware layers. They execute in order — each processes the previous one's output.

In [ ]:
class UppercaseMiddleware(AgentMiddleware):
    def after_agent(self, state):
        state["messages"][-1].content = state["messages"][-1].content.upper()
        return state

stacked_logger = TimedLoggingMiddleware()

agent_stacked = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant.",
    middleware=[stacked_logger, UppercaseMiddleware()],
)

result = agent_stacked.invoke({
    "messages": [HumanMessage(content="Say hello in three languages.")]
})
print(result["messages"][-1].content)

print("\nLogs:")
for log in stacked_logger.logs:
    print(f"  [{log['type']}] {log['content'][:80]}")

## Summary

- `AgentMiddleware` provides four hooks: `before_agent`, `after_agent`, `before_model`, `after_model`
- `@hook_config` configures which message types a hook processes
- `can_jump_to` short-circuits the pipeline for cached or pre-computed responses
- Middleware stacks execute in order — each processes the previous one's output
- Use middleware for logging, caching, content filtering, and output transformation